In [1]:
import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.model_selection import train_test_split
import cv2

# **Step 1: Load and Parse the Output.txt File**
def load_metadata(metadata_file):
    """Reads output.txt and returns a list of filenames with their labels."""
    filenames = []
    labels = []
    
    with open(metadata_file, 'r') as file:
        next(file)  # Skip the header line
        for line in file:
            parts = line.strip().split(",")  # Split by comma
            filename = parts[0].strip()
            label = parts[-1].strip()  # Last column contains the tag
            
            filenames.append(filename)
            labels.append(1 if label == "overlapped" else 0)  # Convert to binary
        
    return filenames, np.array(labels)

# **Step 2: Extract Mel Spectrogram Features**
def preprocess_audio(file_path, img_size=(128, 128)):
    """Converts an audio file into a resized Mel Spectrogram image."""
    y, sr = librosa.load(file_path, sr=22050, duration=3)  # Load audio
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)  # Convert to dB scale
    
    # Resize the spectrogram to a fixed size
    mel_spec_db = cv2.resize(mel_spec_db, img_size)
    return mel_spec_db

# **Step 3: Load Dataset and Preprocess**
AUDIO_DIR = "output_samples"  # Folder containing audio files
METADATA_FILE = "output.txt"  # Metadata file

# Load metadata
filenames, labels = load_metadata(METADATA_FILE)

# Load and process audio files
X = []
for filename in filenames:
    file_path = os.path.join(AUDIO_DIR, filename)
    if os.path.exists(file_path):  # Check if file exists
        spectrogram = preprocess_audio(file_path)
        X.append(spectrogram)

X = np.array(X)
X = X[..., np.newaxis]  # Add channel dimension for CNN input
labels = np.array(labels)

# **Step 4: Split Data into Training and Testing Sets**
X_train, X_test, y_train, y_test = train_test_split(X, labels, test_size=0.2, random_state=42)

# **Step 5: Define CNN Model**
model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 1)),
    MaxPooling2D(2,2),
    
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # Binary classification
])

# **Step 6: Compile and Train the Model**
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

history = model.fit(X_train, y_train, validation_split=0.2, epochs=10, batch_size=32)

# **Step 7: Evaluate the Model**
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {test_acc:.2f}")

# **Step 8: Save the Model**
model.save("speech_overlap_detector.h5")

# **Step 9: Function to Predict Overlap**
def predict_overlap(audio_path):
    """Predict if an audio file is overlapped or not."""
    spectrogram = preprocess_audio(audio_path)
    spectrogram = np.expand_dims(spectrogram, axis=(0, -1))  # Add batch & channel dims
    prediction = model.predict(spectrogram)[0][0]
    return "Overlapped" if prediction > 0.5 else "Not Overlapped"

# Example usage
example_audio = os.path.join(AUDIO_DIR, "audio_4_overlapped.wav")  # Replace with any file
print(predict_overlap(example_audio))


C:\Users\saksh\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 25s 142ms/step - accuracy: 0.5569 - loss: 2.8104 - val_accuracy: 0.7312 - val_loss: 0.5541
Epoch 2/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 22s 136ms/step - accuracy: 0.8083 - loss: 0.4229 - val_accuracy: 0.9539 - val_loss: 0.1283
Epoch 3/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 22s 139ms/step - accuracy: 0.9350 - loss: 0.1716 - val_accuracy: 0.9695 - val_loss: 0.0853
Epoch 4/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 24s 150ms/step - accuracy: 0.9619 - loss: 0.0988 - val_accuracy: 0.9812 - val_loss: 0.0805
Epoch 5/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 24s 147ms/step - accuracy: 0.9702 - loss: 0.0815 - val_accuracy: 0.9891 - val_loss: 0.0324
Epoch 6/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 23s 145ms/step - accuracy: 0.9638 - loss: 0.1009 - val_accuracy: 0.9891 - val_loss: 0.0302
Epoch 7/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 23s 146ms/step - accuracy: 0.9865 - loss: 0.0395 - val_accuracy: 0.9906 - val_loss: 0.0265
Epoch 8/10
160/160 ━━━━━━━━━━━━━━━━━━━━ 24s 147ms/step - accuracy: 0.9903 - loss: 0


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
Overlapped


In [2]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_curve
import numpy as np

# **Step 1: Generate Predictions on the Test Set**
y_pred_prob = model.predict(X_test)  # Get the probabilities
y_pred = (y_pred_prob > 0.5).astype(int)  # Convert probabilities to binary values

# **Step 2: Calculate the Confusion Matrix**
cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()  # Extract True Negatives, False Positives, False Negatives, True Positives

# **Step 3: Calculate Evaluation Metrics**
accuracy = (TP + TN) / (TP + TN + FP + FN)  # Accuracy
FAR = FP / (FP + TN)  # False Alarm Rate
FRR = FN / (FN + TP)  # Miss Rate (False Rejection Rate)

# **Step 4: Calculate EER**
# We will use roc_curve to calculate FAR and FRR at different thresholds
fpr, tpr, thresholds = roc_curve(y_test, y_pred_prob)  # Get FPR and TPR at different thresholds

# EER is the point where FAR and FRR are equal, so we find the threshold where fpr is closest to (1 - tpr)
eer_index = np.argmin(np.abs(fpr - (1 - tpr)))  # Find the index where FPR and TPR are closest
eer = (fpr[eer_index] + (1 - tpr[eer_index])) / 2  # Calculate EER

# **Step 5: Save Results to matrix_cnn.txt**
with open("matrix_cnn.txt", "w") as file:
    file.write("Confusion Matrix:\n")
    file.write(f"{cm}\n\n")
    
    file.write(f"Accuracy: {accuracy:.2f}\n")
    file.write(f"False Alarm Rate (FAR): {FAR:.2f}\n")
    file.write(f"Miss Rate (FRR): {FRR:.2f}\n")
    file.write(f"Equal Error Rate (EER): {eer:.4f}\n")

print("Results saved to matrix_cnn.txt")


50/50 ━━━━━━━━━━━━━━━━━━━━ 2s 36ms/step
Results saved to matrix_cnn.txt
